# Case 3: fixed-epoch asymptotic normality

DPLQR only; tau = 0.5, theta = (1, -1). Compare centering and coverage at
200, 500 and 1000 epochs using paired data/initialization across epoch settings.
Every main and SE projection fit runs its specified epoch count, with no stopping
or best-weight restoration. Validation is monitored only, preserving the pilot's RNG path.
The saved September 10 pilot had a 100-epoch ceiling and patience 15.

Run cells in order with the repository's Python kernel. Increase Q and rerun to
extend completed results. This notebook is supplied unexecuted.


In [ ]:
from pathlib import Path
from types import FunctionType, SimpleNamespace
from contextlib import contextmanager
import ast
import hashlib
import importlib.metadata
import importlib.util
import itertools
import json
import math
import os
import platform
import random
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, skew, kurtosis, probplot
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
from torchtuples import Model
import torchtuples as tt
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dqAux.py").is_file())
EXPERIMENT = ROOT / "results/11092026_asymptotic_normality_case3"
NOTEBOOK = EXPERIMENT / "asymptotic_normality_case3.ipynb"
OUTPUT_DIR = EXPERIMENT / "run"

Q = 20
SAMPLE_SIZES = (500, 1000, 2000, 4000)
EPOCH_COUNTS = (200, 500, 1000)
CASE, TAU = 3, 0.5
THETA_0 = np.array([1.0, -1.0])
BASE_SEED = 20260904
TRAIN_FRACTION = 0.8
HP = dict(depth=2, width=32, batch_size=128, lr=0.005)
THREADS = 1

assert isinstance(Q, int) and Q >= 1
assert CASE == 3 and TAU == 0.5 and np.array_equal(THETA_0, [1.0, -1.0])
assert SAMPLE_SIZES == (500, 1000, 2000, 4000)
assert EPOCH_COUNTS == (200, 500, 1000)
assert TRAIN_FRACTION == 0.8
torch.set_num_threads(THREADS)
torch.use_deterministic_algorithms(True)


Reuse the September 10 source bridge and the September 4 DGP/SE definitions.
The only R call is the existing base-R residual density estimator used for SEs
(set RSCRIPT if Rscript is not detected). Networks and losses come from root dqAux.py.
Z is standardized on training observations; X and Y retain their original scales.
Post-fit nuisance clipping and the full 2×2 covariance estimator follow the pilot.


In [ ]:
BRIDGE_PATH = ROOT / "results/2026-09-10-asymptotic-normality/source_bridge.py"
spec = importlib.util.spec_from_file_location("case3_source_bridge", BRIDGE_PATH)
bridge = importlib.util.module_from_spec(spec)
bytecode_setting = sys.dont_write_bytecode
try:
    sys.dont_write_bytecode = True
    spec.loader.exec_module(bridge)
finally:
    sys.dont_write_bytecode = bytecode_setting

dqNetSparse, covNet = bridge.dqNetSparse, bridge.covNet
checkLoss, checkErrorMean = bridge.checkLoss, bridge.checkErrorMean
REUSED = (
    "seed_all", "nonlinear_truth", "generate_covariates", "generate_dataset",
    "as_numpy", "clip_neural_weights", "tensor_pair", "dplqr_standard_errors",
)
nodes = bridge.selected_source()
science = dict(np=np, norm=norm, random=random, torch=torch, THETA=THETA_0,
               residual_density_zero=bridge.residual_density_zero)
module = ast.Module(body=[nodes[name] for name in REUSED], type_ignores=[])
exec(compile(module, str(bridge.NOTEBOOK), "exec"), science)
original = SimpleNamespace(**{name: science[name] for name in REUSED})


def make_case3(n, rep):
    data_seed = BASE_SEED + CASE * 10_000_000 + n * 1000 + rep
    fit_seed = data_seed + int(round(TAU * 1_000_000))
    rng = np.random.default_rng(data_seed)
    x, z, y, _ = original.generate_dataset(n, 3, rng)
    order = rng.permutation(n)
    tr, va = order[:int(TRAIN_FRACTION * n)], order[int(TRAIN_FRACTION * n):]
    scaler = StandardScaler().fit(z[tr])
    digest = hashlib.sha256(b"".join(a.tobytes() for a in (x, z, y, order))).hexdigest()
    return SimpleNamespace(
        n=n, rep=rep, data_seed=data_seed, fit_seed=fit_seed, data_sha256=digest,
        x_train=x[tr], z_train=z[tr], y_train=y[tr],
        x_val=x[va], z_val=z[va], y_val=y[va], scaler=scaler,
    )


Each epoch setting starts from the same seeds and gets a fresh optimizer.
Both auxiliary projections use that setting's epoch count, as the original
SE routine shares the main fit's profile. Epoch comparisons therefore include
changes in both coefficient and projection optimization.


In [ ]:
class EpochCounter(tt.callbacks.Callback):
    """Count complete epochs and batches; never select or restore weights."""
    def __init__(self, epochs, n_train):
        self.requested = epochs
        self.expected_updates = epochs * math.ceil(n_train / HP["batch_size"])

    def on_fit_start(self):
        self.epochs_run = self.updates = 0
        return False

    def on_batch_end(self):
        if not torch.isfinite(self.model.batch_loss).item():
            raise FloatingPointError("Non-finite training loss")
        self.updates += 1
        return False

    def on_epoch_end(self):
        self.epochs_run += 1
        return False

    def on_fit_end(self):
        if self.epochs_run != self.requested or self.updates != self.expected_updates:
            raise RuntimeError("Fit did not complete its exact epoch/update budget")
        return False


def fit_fixed(net, loss, train_x, train_y, val_x, val_y, epochs):
    model = Model(net, loss, device="cpu")
    model.optimizer.set_lr(HP["lr"])
    counter = EpochCounter(epochs, len(train_y))
    log = model.fit(
        train_x, torch.tensor(train_y[:, None], dtype=torch.float32),
        batch_size=HP["batch_size"], epochs=epochs, callbacks=[counter],
        verbose=False, shuffle=True, num_workers=0,
        val_data=(val_x, torch.tensor(val_y[:, None], dtype=torch.float32)),
        val_batch_size=HP["batch_size"],
    ).to_pandas()
    if len(log) != epochs:
        raise RuntimeError("Training log has an unexpected epoch count")
    details = dict(epochs_run=counter.epochs_run, updates=counter.updates,
                   last_epoch_train_loss=float(log["train_loss"].iloc[-1]),
                   last_epoch_validation_loss=float(log["val_loss"].iloc[-1]))
    if not np.isfinite(list(details.values())).all():
        raise FloatingPointError("Non-finite fit diagnostics")
    return model, details


Atomic checkpoints retain each completed network and result under its
(n, epochs, replication) key. An unfinished network restarts from its seed.
A run fingerprint prevents mixing incompatible settings; Q can increase.
If a kernel is killed, remove its run/.run.lock only after it has exited.


In [ ]:
def atomic_write(path, value, kind="json"):
    temporary = path.with_suffix(path.suffix + ".tmp")
    if kind == "torch":
        torch.save(value, temporary)
    elif kind == "csv":
        value.to_csv(temporary, index=False)
    else:
        temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)


@contextmanager
def run_lock(output):
    path = output / ".run.lock"
    try:
        descriptor = os.open(path, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    except FileExistsError as exc:
        raise RuntimeError(f"Run lock exists: {path}; check its process before removing it") from exc
    try:
        with os.fdopen(descriptor, "w") as stream:
            json.dump(dict(pid=os.getpid(), host=platform.node()), stream)
        yield
    finally:
        path.unlink(missing_ok=True)


def run_identity():
    notebook = json.loads(NOTEBOOK.read_text(encoding="utf-8"))
    implementation = "".join(
        "".join(cell["source"]) for cell in notebook["cells"]
        if "experiment_code" in cell.get("metadata", {}).get("tags", [])
    )
    sources = (ROOT / "dqAux.py", BRIDGE_PATH, bridge.NOTEBOOK,
               BRIDGE_PATH.parent / "density_only.R")
    r_version = subprocess.run([bridge.find_rscript(), "--version"], capture_output=True,
                               text=True, check=True, timeout=30)
    return dict(
        schema=1, case=CASE, tau=TAU, theta=THETA_0.tolist(), seed=BASE_SEED,
        train_fraction=TRAIN_FRACTION, hp=HP, threads=THREADS,
        deterministic_algorithms=True, early_stopping=False,
        projection_epochs="same as main", validation="monitor only", clipping="post-fit nuisance",
        implementation_sha256=hashlib.sha256(implementation.encode()).hexdigest(),
        sources={str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in sources},
        versions={name: importlib.metadata.version(name) for name in
                  ("numpy", "pandas", "scipy", "scikit-learn", "torch", "torchtuples")},
        python=platform.python_version(), platform=platform.platform(),
        r_version=(r_version.stdout + r_version.stderr).strip(),
    )


def replication_directory(output, n, epochs, rep):
    return output / "replications" / f"n_{n}_epochs_{epochs}_rep_{rep:06d}"


def fit_replication(data, epochs, directory, fingerprint):
    directory.mkdir(parents=True, exist_ok=True)
    setting = dict(n=data.n, epochs=epochs, rep=data.rep)
    stage_path = directory / "fits.pt"
    if stage_path.exists():
        bundle = torch.load(stage_path, map_location="cpu", weights_only=True)
        if (bundle["fingerprint"] != fingerprint or bundle["setting"] != setting
                or bundle["data_sha256"] != data.data_sha256):
            raise ValueError(f"Incompatible checkpoint: {stage_path}")
    else:
        bundle = dict(fingerprint=fingerprint, setting=setting,
                      data_sha256=data.data_sha256, stages={})

    def stage(name, seed, net_factory, loss, train_x, train_y, val_x, val_y, clip=False):
        original.seed_all(seed)
        net = net_factory()
        saved = bundle["stages"].get(name)
        if saved is None:
            model, details = fit_fixed(net, loss, train_x, train_y, val_x, val_y, epochs)
            if clip:
                original.clip_neural_weights(model.net)
            saved = dict(state=model.net.state_dict(), seed=seed, **details)
            bundle["stages"][name] = saved
            atomic_write(stage_path, bundle, "torch")
        else:
            if (saved["seed"] != seed or saved["epochs_run"] != epochs
                    or saved["updates"] != epochs * math.ceil(len(train_y) / HP["batch_size"])):
                raise ValueError(f"Incomplete or incompatible stage: {stage_path} / {name}")
            net.load_state_dict(saved["state"])
            model = Model(net, loss, device="cpu")
        return model, saved

    def main_net():
        net = dqNetSparse(2, 8, torch.zeros((1, 2), dtype=torch.float32),
                          [HP["depth"], HP["width"]], sparseRatio=0.5)
        net.linLinear.reset_parameters()
        return net

    train_x = original.tensor_pair(data.x_train, data.scaler.transform(data.z_train))
    val_x = original.tensor_pair(data.x_val, data.scaler.transform(data.z_val))
    model, main = stage("main", data.fit_seed, main_net, checkLoss(tau=TAU),
                        train_x, data.y_train, val_x, data.y_val, clip=True)

    def cached_projection(z_train, target_train, z_val, target_val, hp, seed, binary):
        column = 1 if binary else 2
        projection, saved = stage(
            f"projection_{column}", seed,
            lambda: covNet(8, [hp["depth"], hp["width"]], logic=binary),
            nn.MSELoss(), torch.tensor(z_train, dtype=torch.float32), target_train,
            torch.tensor(z_val, dtype=torch.float32), target_val,
        )
        return projection, SimpleNamespace(epochs_run=saved["epochs_run"])

    # Reuse the full SE formula with only the projection fitter replaced.
    se_function = FunctionType(
        original.dplqr_standard_errors.__code__,
        dict(science, fit_projection=cached_projection), "fixed_epoch_standard_errors",
    )
    se, density, projection_epochs = se_function(
        model, data.scaler, dict(HP, epochs=epochs),
        data.x_train, data.z_train, data.y_train,
        data.x_val, data.z_val, data.y_val, TAU, data.fit_seed + 500_000,
    )
    theta = model.net.linLinear.weight.detach().cpu().numpy().reshape(-1).astype(float)
    if not np.isfinite(theta).all() or not np.isfinite(se).all() or np.any(se <= 0):
        raise FloatingPointError("Invalid coefficient or SE")
    if projection_epochs != [epochs, epochs]:
        raise RuntimeError("Projection epoch count mismatch")
    n_train = len(data.y_train)
    errors = theta - THETA_0
    lower, upper = theta - norm.ppf(0.975) * se, theta + norm.ppf(0.975) * se
    prediction = original.as_numpy(model.predict(train_x)).reshape(-1, 1)
    row = dict(
        case=3, method="DPLQR", tau=TAU, **setting, n_train=n_train, n_val=len(data.y_val),
        data_seed=data.data_seed, fit_seed=data.fit_seed, data_sha256=data.data_sha256,
        epochs_run=main["epochs_run"], optimizer_updates=main["updates"],
        projection_1_epochs_run=projection_epochs[0], projection_2_epochs_run=projection_epochs[1],
        projection_1_seed=data.fit_seed + 500_000, projection_2_seed=data.fit_seed + 500_001,
        density_zero=float(density), last_epoch_train_loss=main["last_epoch_train_loss"],
        last_epoch_validation_loss=main["last_epoch_validation_loss"],
        training_loss=float(checkErrorMean(prediction, data.y_train[:, None], tau=TAU)),
    )
    for j in (1, 2):
        i = j - 1
        row.update({
            f"theta_hat_{j}": float(theta[i]), f"theta_0_{j}": float(THETA_0[i]),
            f"estimated_se_{j}": float(se[i]), f"theta_error_{j}": float(errors[i]),
            f"root_n_train_error_{j}": float(np.sqrt(n_train) * errors[i]),
            f"T_{j}": float(errors[i] / se[i]), f"ci_lower_{j}": float(lower[i]),
            f"ci_upper_{j}": float(upper[i]),
            f"covered_{j}": bool(lower[i] <= THETA_0[i] <= upper[i]),
        })
    if not all(np.isfinite(v) for v in row.values() if isinstance(v, (int, float))):
        raise FloatingPointError("Non-finite replication result")
    atomic_write(directory / "result.json", dict(fingerprint=fingerprint, row=row))
    return row


def collect_results(output, fingerprint, expected):
    rows = []
    for path in sorted((output / "replications").glob("n_*_epochs_*_rep_*/result.json")):
        saved = json.loads(path.read_text(encoding="utf-8"))
        row = saved["row"]
        key = (row["n"], row["epochs"], row["rep"])
        if saved["fingerprint"] != fingerprint or key not in expected:
            raise ValueError(f"Incompatible result or reduced grid/Q: {path}")
        if path.parent != replication_directory(output, *key):
            raise ValueError(f"Result key/path mismatch: {path}")
        rows.append(row)
    if len({(r["n"], r["epochs"], r["rep"]) for r in rows}) != len(rows):
        raise ValueError("Duplicate replication keys")
    return rows


In [ ]:
def run_experiment():
    output = OUTPUT_DIR.resolve()
    if EXPERIMENT.resolve() not in output.parents:
        raise ValueError("OUTPUT_DIR must be a subdirectory of this experiment")
    output.mkdir(parents=True, exist_ok=True)
    identity = run_identity()
    fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    expected = set(itertools.product(SAMPLE_SIZES, EPOCH_COUNTS, range(1, Q + 1)))
    config_path = output / "run_config.json"

    with run_lock(output):
        if config_path.exists():
            old = json.loads(config_path.read_text(encoding="utf-8"))
            if old["identity"] != identity:
                raise ValueError("Configuration/source changed; choose a fresh OUTPUT_DIR")
            if Q < old["requested_Q"]:
                raise ValueError("Retain or increase Q when resuming this output directory")
        elif (output / "replications").exists():
            raise ValueError("Checkpoints exist without a run identity")
        rows = collect_results(output, fingerprint, expected)
        completed = {(r["n"], r["epochs"], r["rep"]) for r in rows}
        metadata = dict(identity=identity, fingerprint=fingerprint, requested_Q=Q,
                        sample_sizes=list(SAMPLE_SIZES), epoch_counts=list(EPOCH_COUNTS),
                        expected_rows=len(expected))

        def checkpoint(status, error=None):
            if rows:
                atomic_write(output / "raw_replications.csv",
                             pd.DataFrame(rows).sort_values(["n", "epochs", "rep"]), "csv")
            metadata.update(status=status, completed_rows=len(rows), error=error)
            atomic_write(config_path, metadata)

        checkpoint("running")
        try:
            for n in SAMPLE_SIZES:
                for rep in range(1, Q + 1):
                    missing = [e for e in EPOCH_COUNTS if (n, e, rep) not in completed]
                    if not missing:
                        continue
                    data = make_case3(n, rep)
                    for epochs in missing:
                        row = fit_replication(data, epochs,
                                              replication_directory(output, n, epochs, rep), fingerprint)
                        rows.append(row)
                        completed.add((n, epochs, rep))
                        checkpoint("running")
                        print(f"Saved n={n}, epochs={epochs}, rep={rep}; {len(rows)}/{len(expected)}")
            if completed != expected:
                raise RuntimeError("Incomplete experiment grid")
            checkpoint("complete")
        except BaseException as exc:
            checkpoint("failed", repr(exc))
            raise
    return pd.DataFrame(rows).sort_values(["n", "epochs", "rep"]).reset_index(drop=True)


The next cell runs the simulation when explicitly executed. To regenerate
reports later without fitting, skip it and run the summary and plotting cells.


In [ ]:
raw = run_experiment()


In [ ]:
# Read only completed results; summaries include their actual replication counts.
output = OUTPUT_DIR.resolve()
if EXPERIMENT.resolve() not in output.parents:
    raise ValueError("OUTPUT_DIR must be a subdirectory of this experiment")
metadata = json.loads((output / "run_config.json").read_text(encoding="utf-8"))
expected = set(itertools.product(SAMPLE_SIZES, EPOCH_COUNTS, range(1, Q + 1)))
raw = pd.DataFrame(collect_results(output, metadata["fingerprint"], expected))
if raw.empty:
    raise ValueError("No completed replications to summarize")
raw = raw.sort_values(["n", "epochs", "rep"]).reset_index(drop=True)
records = []
for n, epochs in itertools.product(SAMPLE_SIZES, EPOCH_COUNTS):
    group = raw.loc[(raw["n"] == n) & (raw["epochs"] == epochs)]
    if group.empty:
        continue
    if group["n_train"].nunique() != 1:
        raise ValueError("Mixed training sample sizes in a setting")
    n_train = int(group["n_train"].iloc[0])
    for j in (1, 2):
        errors = group[f"theta_error_{j}"].to_numpy()
        t_values = group[f"T_{j}"].to_numpy()
        q025, q50, q975 = np.quantile(t_values, [0.025, 0.5, 0.975], method="linear")
        empirical_sd = float(group[f"theta_hat_{j}"].std(ddof=1))
        records.append(dict(
            n=n, epochs=epochs, coefficient=j, n_train=n_train, Q_completed=len(group),
            Q_requested=Q, complete=len(group) == Q,
            bias=float(errors.mean()), root_n_train_scaled_bias=float(np.sqrt(n_train) * errors.mean()),
            empirical_sd=empirical_sd, root_n_train_scaled_sd=float(np.sqrt(n_train) * empirical_sd),
            mean_estimated_se=float(group[f"estimated_se_{j}"].mean()),
            mean_T=float(t_values.mean()), sd_T=float(pd.Series(t_values).std(ddof=1)),
            skewness_T=float(skew(t_values, bias=False)) if len(group) >= 3 and np.ptp(t_values) > 0 else np.nan,
            excess_kurtosis_T=float(kurtosis(t_values, fisher=True, bias=False))
                              if len(group) >= 4 and np.ptp(t_values) > 0 else np.nan,
            T_q025=float(q025), T_q50=float(q50), T_q975=float(q975),
            coverage_95=float(group[f"covered_{j}"].mean()),
            mean_training_loss=float(group["training_loss"].mean()),
        ))
summary = pd.DataFrame(records)
atomic_write(output / "raw_replications.csv", raw, "csv")
atomic_write(output / "summary.csv", summary, "csv")
display(summary.loc[summary["coefficient"] == 1].round(4))


In [ ]:
figures = output / "figures"
figures.mkdir(parents=True, exist_ok=True)
primary = summary.loc[summary["coefficient"] == 1]
fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
panels = [
    ("mean_T", "Mean T1", 0),
    ("sd_T", "SD(T1)", 1),
    ("coverage_95", "95% CI coverage", 0.95),
    ("root_n_train_scaled_bias", r"$\sqrt{n_{train}}$-scaled bias of $\hat\theta_1$", 0),
]
for ax, (metric, label, reference) in zip(axes.flat, panels):
    for epochs in EPOCH_COUNTS:
        group = primary.loc[primary["epochs"] == epochs].sort_values("n")
        ax.plot(group["n"], group[metric], marker="o", label=f"{epochs} epochs")
    ax.axhline(reference, color="black", ls="--", lw=1)
    ax.set(xlabel="Total generated n", ylabel=label, xticks=SAMPLE_SIZES)
    ax.grid(alpha=0.2)
    ax.legend()
fig.suptitle(f"Case 3, theta1: Q requested = {Q}; see summary for completed counts")
fig.savefig(figures / "theta1_by_sample_size.png", dpi=180, bbox_inches="tight")
plt.show()

largest_n = 4000
values = {
    e: raw.loc[(raw["n"] == largest_n) & (raw["epochs"] == e), "T_1"].to_numpy()
    for e in EPOCH_COUNTS
}
available = [v for v in values.values() if len(v)]
if not available:
    raise ValueError("No completed n=4000 replications for distribution plots")
pooled = np.concatenate(available)
lo, hi = min(-4.0, float(pooled.min())), max(4.0, float(pooled.max()))
padding = 0.04 * (hi - lo)
lo, hi = lo - padding, hi + padding
bins = np.linspace(lo, hi, max(7, int(np.sqrt(max(map(len, available)))) + 1))
grid = np.linspace(lo, hi, 500)
fig, axes = plt.subplots(1, len(EPOCH_COUNTS), figsize=(13, 4), sharex=True, sharey=True,
                         constrained_layout=True)
for ax, epochs in zip(np.atleast_1d(axes), EPOCH_COUNTS):
    t_values = values[epochs]
    if len(t_values):
        ax.hist(t_values, bins=bins, density=True, alpha=0.65, edgecolor="white", label="T1")
    ax.plot(grid, norm.pdf(grid), color="black", label="N(0,1)")
    ax.set(title=f"{epochs} epochs (Q={len(t_values)})", xlabel="T1", xlim=(lo, hi))
    ax.legend()
axes[0].set_ylabel("Density")
fig.suptitle(f"Case 3, n={largest_n}: studentized theta1")
fig.savefig(figures / "theta1_histograms_n4000.png", dpi=180, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, len(EPOCH_COUNTS), figsize=(13, 4), sharex=True, sharey=True,
                         constrained_layout=True)
for ax, epochs in zip(np.atleast_1d(axes), EPOCH_COUNTS):
    t_values = values[epochs]
    if len(t_values) >= 2:
        theoretical, ordered = probplot(t_values, dist="norm", fit=False)
        ax.scatter(theoretical, ordered, s=22)
    ax.plot([lo, hi], [lo, hi], color="black", ls="--", label="N(0,1): y=x")
    ax.set(title=f"{epochs} epochs (Q={len(t_values)})", xlabel="N(0,1) quantile",
           xlim=(-3.5, 3.5), ylim=(lo, hi))
    ax.grid(alpha=0.2)
    ax.legend()
axes[0].set_ylabel("Observed T1 quantile")
fig.suptitle(f"Case 3, n={largest_n}: normal QQ plots")
fig.savefig(figures / "theta1_qq_n4000.png", dpi=180, bbox_inches="tight")
plt.show()


Assess whether mean T1 approaches 0, SD(T1) approaches 1, and coverage approaches
0.95 as epochs increase, alongside scaled bias and the n=4000 distributions.
Q=20 supports a pilot comparison; use larger Q for stable tail and coverage estimates.
Training loss is the final clipped model's full training check loss;
last_epoch_train_loss is the optimizer's last epoch minibatch loss.
